# Teach a vision model to read math — GRPO + OpenEnv

This notebook runs the same recipe as `01-latex-ocr/train/grpo_latex_ocr.py` and HF Jobs:
serve images, evaluate Qwen3-VL, train LoRA with GRPO, evaluate again, and optionally publish the model.
The environment, reward, and runner are maintained together in [HuggingEnvs](https://github.com/adithya-s-k/HuggingEnvs/tree/main/01-latex-ocr).

Use a CUDA GPU. Start with the two-step smoke run; the full defaults use eight generations per group.


## 1. Install from this repository

In a checkout, this uses your local source. In Colab, it clones HuggingEnvs.
Set `HUGGINGENVS_REVISION` to a commit SHA to reproduce a particular run.
The dependency versions come from the checked-in environment lockfile. Restart the kernel if packages were already imported.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

roots = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in roots if (p / "01-latex-ocr/train/grpo_latex_ocr.py").is_file()), None)
if REPO is None:
    REPO = Path(tempfile.mkdtemp()) / "HuggingEnvs"
    subprocess.run(["git", "clone", "https://github.com/adithya-s-k/HuggingEnvs.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", os.getenv("HUGGINGENVS_REVISION", "main")], check=True)
PROJECT = REPO / "01-latex-ocr"
ENV_PROJECT = PROJECT / "envs/latex_ocr"
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
requirements = Path(tempfile.mkdtemp()) / "requirements.txt"
subprocess.run([sys.executable, "-m", "uv", "export", "--project", str(ENV_PROJECT),
                "--frozen", "--extra", "train", "--no-dev", "--no-emit-project",
                "--output-file", str(requirements)], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ENV_PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "train"))
print("Source:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


## 2. Settings

The default starts a CPU environment beside the GPU trainer, in indexed `materialize` mode, and stops it afterwards.
To use the existing hosted environment, set `env_url="https://adithyask-latex-ocr-env.hf.space"`.
Hosted training needs enough concurrent sessions for the generations plus evaluation.

No login is needed for the public dataset and model. Log in with a write token only if enabling Trackio or publishing.


In [ ]:
from grpo_latex_ocr import Config, run

SMOKE = True
config = Config(
    model="Qwen/Qwen3-VL-2B-Instruct",
    env_url="",
    max_steps=2 if SMOKE else 30,
    num_generations=2 if SMOKE else 8,
    eval_samples=2 if SMOKE else 50,
    output_dir=str(PROJECT / "results/local-run"),
    trackio_space="",   # Optional: "your-name/trackio-latex-ocr"
    push_repo_id="",    # Optional: "your-name/qwen3-vl-2b-latex-ocr-grpo"
    smoke=SMOKE,
)
config


## 3. Check the environment first

This CPU check creates two tiny image fixtures and exercises HTTP task discovery, independent WebSocket sessions,
PNG decoding, exact/partial/empty/padded rewards, and stream exhaustion/restart.
Add `--real-data` to use two actual `unsloth/LaTeX_OCR` samples per split.


In [ ]:
subprocess.run([sys.executable, "-m", "latex_ocr_env.smoke"], check=True)


## 4. Evaluate → train → evaluate

The environment returns each image and keeps its reference LaTeX hidden until the episode ends.
Every completion in a GRPO group sees the same indexed image. Raw model output goes to the server unchanged,
so the scorer can penalize excessive whitespace as well as compute edit similarity and exact match.

`run` evaluates the first N test images before and after training, saves the adapter and processor, and closes all sessions.
A smoke run checks finite loss, completed optimizer steps, and an actual adapter weight update.
Two steps are a pipeline check; they do not establish a quality improvement.


In [ ]:
summary = run(config)
print(f"Before: {summary['baseline']['mean_reward']:.4f}")
print(f"After:  {summary['trained']['mean_reward']:.4f}")
print(f"Delta:  {summary['reward_delta']:+.4f}")
print("Saved to:", config.output_dir)


## 5. Inspect predictions

The JSON files retain task indices, raw predictions, targets, and rewards so the aggregate score can be checked.


In [ ]:
for before, after in zip(summary["baseline"]["samples"], summary["trained"]["samples"]):
    print("Task", before["index"], "target:", before["target"])
    print("Before:", repr(before["prediction"]), "reward:", before["reward"])
    print("After: ", repr(after["prediction"]), "reward:", after["reward"])
    print()


## Run unattended on HF Jobs

From a pushed checkout:

```bash
hf jobs uv run --flavor a10g-small --timeout 30m --secrets HF_TOKEN \
  01-latex-ocr/train/hf_job.py --revision "$(git rev-parse HEAD)" --smoke
```

For a full run and persistent bucket output, see [the training README](../train/README.md).
The Jobs entry point downloads that exact revision and uses the same locked dependencies and runner as this notebook.
